## 核心区别

|        | 真实微批次（`mb`）                          | 虚拟微批次（`v_id`）                    |
| ------ | ------------------------------------ | -------------------------------- |
| **本质** | 数据的实际切分                              | 执行的顺序编号                          |
| **数量** | `M = total_batch / micro_batch_size` | `M × VPP`（通常更多）                  |
| **含义** | "哪一份数据"                              | "第几步执行"                          |
| **范围** | `0 ~ M-1`                            | `0 ~ (M × num_model_chunks - 1)` |


## 为什么需要虚拟微批次？
### 问题：VPP 一个 GPU 有多个虚拟阶段

In [ ]:
配置: PP=2, VPP=2, M=5（5个真实微批次）

真实微批次数据:
mb0, mb1, mb2, mb3, mb4  ← 只有5份数据

但每个数据需要经过 2 个 chunk（虚拟阶段）才能算完:
  mb0 → chunk0 → chunk1 → 完成
  mb1 → chunk0 → chunk1 → 完成
  ...

如果只用 mb 编号，无法区分"mb0在chunk0"还是"mb0在chunk1"

### 解决方案：引入虚拟微批次

In [ ]:
mb0 在 chunk0 → v_id=0（第0步执行）
mb0 在 chunk1 → v_id=3（第3步执行）

v_id 是"执行步骤"，mb 是"数据身份"

### 映射关系图解

In [ ]:
调度表:
v_id | 0  1  2  3  4  5  6  7  8  9
mb   | 0  1  2  0  1  2  3  4  3  4
ch   | 0  0  0  1  1  1  0  0  1  1

真实微批次 mb=0 的完整生命周期:
    
  v_id=0: mb=0, ch=0  → 第0步，数据0，跑虚拟阶段0
     ↓ （P2P通信，输出传给下一个stage）
  v_id=3: mb=0, ch=1  → 第3步，数据0，跑虚拟阶段1
     ↓ （计算loss，反向传播）
  v_id=?: mb=0, ch=1  → 反向传播阶段1
  v_id=?: mb=0, ch=0  → 反向传播阶段0

真实微批次 mb=0 被拆成多个 v_id 步骤执行！

### 数量关系

In [ ]:
总虚拟微批次数 = 真实微批次数 × 虚拟阶段数
              = M × VPP
              = 5 × 2 = 10

验证: v_id 从 0 到 9，共 10 个 ✓

## 执行视角 vs 数据视角
### 数据视角（真实微批次）


In [ ]:
mb=0: 数据样本 [batch_size, seq_len] 的第0份
mb=1: 第1份
...

每个 mb 需要完整经过所有层：
  Input → Layer1-2(chunk0) → Layer3-4(chunk1) → ... → Output

### 执行视角（虚拟微批次）
- 关键：同一 mb 的两个 chunk 间隔 G=3 个 v_id 执行，不是连续的！

In [ ]:
GPU 0 (负责 chunk0 和 chunk1) 的执行顺序:

v_id=0: 跑 chunk0, mb=0  ← 先跑所有mb的chunk0（组0）
v_id=1: 跑 chunk0, mb=1
v_id=2: 跑 chunk0, mb=2

v_id=3: 跑 chunk1, mb=0  ← 再跑同样mb的chunk1（组0）
v_id=4: 跑 chunk1, mb=1
v_id=5: 跑 chunk1, mb=2

v_id=6: 跑 chunk0, mb=3  ← 下一组mb的chunk0（组1）
v_id=7: 跑 chunk0, mb=4

v_id=8: 跑 chunk1, mb=3  ← 同样mb的chunk1（组1）
v_id=9: 跑 chunk1, mb=4

## 为什么同一 mb 的 chunk 不连续？
- <font color='red'>间隔 G=3 给通信和流水线填充留出时间。</font>

In [ ]:
如果连续执行（错误设计）:
v_id=0: mb=0, ch=0
v_id=1: mb=0, ch=1  ← 立即执行chunk1，但chunk0的输出还没传到下一个stage！

正确设计（间隔执行）:
v_id=0: mb=0, ch=0  → 输出通过P2P发送
v_id=1: mb=1, ch=0  → 同时，下一个stage在算mb=0的chunk0
v_id=2: mb=2, ch=0  → 继续填充流水线
    
v_id=3: mb=0, ch=1  ← 此时mb=0的chunk0已到达，可以执行chunk1

## 代码中的使用
### 查表获取信息

In [ ]:
# 已知 v_id，查 mb 和 ch
v_id = 3
mb = microbatch_id_table[v_id]      # → 0
ch = model_chunk_id_table[v_id]     # → 1

# 含义：第3步执行，用数据0，跑虚拟阶段1

### 反向传播时的特殊映射
- 反向传播方向：<font color='red'>从输出端（ch=1）往输入端（ch=0）传播。</font>
  - 从最后一层向第一层传播

In [ ]:
# 正向：v_id=3 → mb=0, ch=1
# 反向：同样是mb=0，但ch要颠倒

# 反向调度表:
v_id | 0  1  2  3  4  5  6  7  8  9
mb   | 0  1  2  0  1  2  3  4  3  4
ch   | 1  1  1  0  0  0  1  1  0  0  ← ch=1的先反，ch=0的后反

# mb=0 的反向：
#   v_id=0: mb=0, ch=1  → 先反chunk1（靠近输出端）
#   v_id=3: mb=0, ch=0  → 后反chunk0（靠近输入端）

## 一句话总结
- 真实微批次（mb）是"数据的身份证"——哪一份数据；
- 虚拟微批次（v_id）是"执行的序号"——第几步跑什么。
- 一个真实微批次需要经过多个虚拟阶段（chunk），所以被拆成多个虚拟微批次步骤执行。
- 调度表的核心作用就是:
  - <font color='red'>建立"执行顺序"到"数据身份+阶段位置"的映射，让VPP交错调度正确运转。</font>